# Inspeccion de Datos - Practica 6 ETL

Este notebook realiza la inspeccion inicial de `Catalog_Orders.txt`, `Web_orders.txt` y `products.txt`.

Flujo:
1. Carga y perfilado inicial.
2. Calidad de datos y deteccion de errores (en especial `CATALOG`).
3. Estandarizacion de campos clave (fecha, catalogo, pcode, cliente).
4. Re-analisis post-limpieza y export de datos limpios para ETL.


In [13]:
import re
from difflib import get_close_matches

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

CANONICAL_CATALOGS = ['Sports', 'Pets', 'Toys', 'Gardening', 'Software', 'Collectibles']


In [14]:
def read_catalog_orders(path='Catalog_Orders.txt'):
    df = pd.read_csv(path)
    return df


def read_web_orders(path='Web_orders.txt'):
    # El archivo web tiene encabezado con comas, pero registros con ';' y columnas en orden diferente.
    names = ['ID', 'INV', 'PCODE', 'DATE', 'CATALOG', 'QTY', 'custnum']
    df = pd.read_csv(path, sep=';', quotechar='"', header=None, skiprows=1, names=names)
    return df


def read_products(path='products.txt'):
    df = pd.read_csv(path)
    return df


catalog_df = read_catalog_orders()
web_df = read_web_orders()
products_df = read_products()

print('Catalog_Orders:', catalog_df.shape)
print('Web_orders:', web_df.shape)
print('Products:', products_df.shape)


Catalog_Orders: (6767, 7)
Web_orders: (943, 7)
Products: (192, 7)


In [15]:
display(catalog_df.head(3))
display(web_df.head(3))
display(products_df.head(3))


,ID,INV,DATE,CATALOG,PCODE,QTY,custnum
0,1,107707.0,3/97/7 00:00:00,Sports,SP1000,1.00,1242.0
1,2,110633.0,3/97/25 00:00:00,Gardening,GD2200,1.00,1243.0
2,3,111155.0,3/97/28 00:00:00,Pets,PT1300,3.00,1244.0


,ID,INV,PCODE,DATE,CATALOG,QTY,custnum
0,1,2513000.0,GD3200,17/12/2000 00:00:00,Gardening,3.0,"Rifkin, Mr. Bob"
1,2,2513000.0,PT1400,6/1/2001 00:00:00,Pets,1.0,"Rifkin, Mr. Bob"
2,3,2513000.0,PT2000,6/1/2001 00:00:00,Pet,1.0,"Rifkin, Mr. Bob"


,ID,TYPE,DESCRIP,PRICE,COST,PCODE,supplier
0,1,Database,Track-It-All,560.00,425.0,SW1005,"Software America, Inc."
1,2,Database,Omnibus,499.95,450.0,SW1207,Software America
2,3,Database,Balboa,499.00,300.0,SW3049,Software America


In [16]:
def profile_df(df, name):
    print(f'===== {name} =====')
    print('Tipos de datos:')
    print(df.dtypes)
    print('Nulos por columna:')
    print(df.isna().sum())
    print('Duplicados exactos:', df.duplicated().sum())
    print('Cardinalidad (nunique):')
    print(df.nunique(dropna=False))
    print('')


profile_df(catalog_df, 'Catalog_Orders')
profile_df(web_df, 'Web_orders')
profile_df(products_df, 'Products')


===== Catalog_Orders =====
Tipos de datos:
ID           int64
INV        float64
DATE           str
CATALOG        str
PCODE          str
QTY            str
custnum    float64
dtype: object
Nulos por columna:
ID         0
INV        0
DATE       0
CATALOG    2
PCODE      0
QTY        6
custnum    1
dtype: int64
Duplicados exactos: 0
Cardinalidad (nunique):
ID         6767
INV        1711
DATE       1019
CATALOG      26
PCODE       226
QTY           7
custnum    3164
dtype: int64

===== Web_orders =====
Tipos de datos:
ID           int64
INV        float64
PCODE          str
DATE           str
CATALOG        str
QTY        float64
custnum        str
dtype: object
Nulos por columna:
ID         0
INV        0
PCODE      0
DATE       0
CATALOG    0
QTY        3
custnum    0
dtype: int64
Duplicados exactos: 0
Cardinalidad (nunique):
ID         943
INV        160
PCODE      119
DATE       331
CATALOG     10
QTY          5
custnum    166
dtype: int64

===== Products =====
Tipos de datos:
ID  

In [17]:
def top_values(df, column, n=15):
    return df[column].astype(str).str.strip().value_counts(dropna=False).head(n)

print('Distribucion CATALOG (Catalog_Orders):')
display(top_values(catalog_df, 'CATALOG', 20))

print('Distribucion CATALOG (Web_orders):')
display(top_values(web_df, 'CATALOG', 20))

print('Distribucion PCODE invalidos respecto a Products:')
valid_pcodes = set(products_df['PCODE'].astype(str).str.upper().str.strip())
invalid_catalog = (~catalog_df['PCODE'].astype(str).str.upper().str.strip().isin(valid_pcodes)).sum()
invalid_web = (~web_df['PCODE'].astype(str).str.upper().str.strip().isin(valid_pcodes)).sum()
print('Catalog_Orders - PCODE fuera de maestro:', int(invalid_catalog))
print('Web_orders - PCODE fuera de maestro:', int(invalid_web))


Distribucion CATALOG (Catalog_Orders):


CATALOG
Sports          1851
Pets            1671
Toys            1309
Gardening        933
Software         506
Collectibles     150
Sport             59
Toy               52
Softwares         31
Tosy              27
Softwars          25
Collectible       23
Sporst            23
Spots             19
Pest              18
Softwar           15
GARDENING         14
Gardning          11
Pet                9
Colectibles        6
Name: count, dtype: int64

Distribucion CATALOG (Web_orders):


CATALOG
Sports        338
Toys          234
Gardening     191
Pets          172
Sport           2
Toy             2
Pet             1
Tosy            1
Gardenings      1
Garden          1
Name: count, dtype: int64

Distribucion PCODE invalidos respecto a Products:
Catalog_Orders - PCODE fuera de maestro: 129
Web_orders - PCODE fuera de maestro: 26


In [18]:
def parse_catalog_date(value):
    # Formato esperado: M/YY/D HH:MM:SS (ej: 3/97/7 00:00:00)
    s = str(value).strip()
    m = re.match(r'^(\d{1,2})/(\d{2})/(\d{1,2})', s)
    if not m:
        return pd.NaT

    month = int(m.group(1))
    yy = int(m.group(2))
    day = int(m.group(3))
    year = 2000 + yy if yy <= 30 else 1900 + yy

    try:
        return pd.Timestamp(year=year, month=month, day=day)
    except ValueError:
        return pd.NaT


def parse_web_date(value):
    # Formato observado: DD/MM/YYYY HH:MM:SS
    return pd.to_datetime(value, dayfirst=True, errors='coerce')


catalog_date_parsed = catalog_df['DATE'].apply(parse_catalog_date)
web_date_parsed = web_df['DATE'].apply(parse_web_date)

print('Catalog_Orders - fechas parseadas:', float(catalog_date_parsed.notna().mean()))
print('Web_orders - fechas parseadas:', float(web_date_parsed.notna().mean()))


Catalog_Orders - fechas parseadas: 1.0
Web_orders - fechas parseadas: 1.0


In [19]:
CATALOG_MAP = {
    'sport': 'Sports', 'sports': 'Sports', 'sprots': 'Sports', 'sporst': 'Sports', 'sposts': 'Sports', 'spots': 'Sports',
    'pet': 'Pets', 'pets': 'Pets', 'pest': 'Pets',
    'toy': 'Toys', 'toys': 'Toys', 'tosy': 'Toys',
    'gardening': 'Gardening', 'gardenings': 'Gardening', 'garden': 'Gardening', 'gardning': 'Gardening',
    'software': 'Software', 'softwares': 'Software', 'softwars': 'Software', 'softwar': 'Software',
    'collectible': 'Collectibles', 'collectibles': 'Collectibles', 'colectibles': 'Collectibles'
}


def normalize_catalog(value):
    s = str(value).strip().lower()
    if s in CATALOG_MAP:
        return CATALOG_MAP[s]

    # Fallback por similitud
    candidates = get_close_matches(s, [k for k in CATALOG_MAP.keys()], n=1, cutoff=0.75)
    if candidates:
        return CATALOG_MAP[candidates[0]]
    return pd.NA


def normalize_pcode(value, valid_codes):
    s = str(value).strip().upper().replace(' ', '')
    # Corrige O por 0 en parte numerica
    m = re.match(r'^([A-Z]+)([A-Z0-9]+)$', s)
    if m:
        prefix, suffix = m.groups()
        s = prefix + suffix.replace('O', '0')

    if s in valid_codes:
        return s

    # Fallback por similitud (1 posible candidato)
    candidates = get_close_matches(s, list(valid_codes), n=1, cutoff=0.8)
    if candidates:
        return candidates[0]
    return pd.NA


def normalize_customer(raw_value):
    if pd.isna(raw_value):
        return pd.NA
    s = str(raw_value).strip()
    s = re.sub(r'\s+', ' ', s)
    return s


In [20]:
valid_pcodes = set(products_df['PCODE'].astype(str).str.upper().str.strip())

catalog_clean = catalog_df.copy()
catalog_clean['source_channel'] = 'Catalog'
catalog_clean['order_date'] = catalog_clean['DATE'].apply(parse_catalog_date)
catalog_clean['CATALOG_RAW'] = catalog_clean['CATALOG']
catalog_clean['CATALOG'] = catalog_clean['CATALOG'].apply(normalize_catalog)
catalog_clean['PCODE_RAW'] = catalog_clean['PCODE']
catalog_clean['PCODE'] = catalog_clean['PCODE'].apply(lambda x: normalize_pcode(x, valid_pcodes))
catalog_clean['customer_nk'] = catalog_clean['custnum'].apply(normalize_customer)

web_clean = web_df.copy()
web_clean['source_channel'] = 'Web'
web_clean['order_date'] = web_clean['DATE'].apply(parse_web_date)
web_clean['CATALOG_RAW'] = web_clean['CATALOG']
web_clean['CATALOG'] = web_clean['CATALOG'].apply(normalize_catalog)
web_clean['PCODE_RAW'] = web_clean['PCODE']
web_clean['PCODE'] = web_clean['PCODE'].apply(lambda x: normalize_pcode(x, valid_pcodes))
web_clean['customer_nk'] = web_clean['custnum'].apply(normalize_customer)

orders_clean = pd.concat([catalog_clean, web_clean], ignore_index=True)

# Normaliza tipos numericos
orders_clean['ID'] = pd.to_numeric(orders_clean['ID'], errors='coerce').astype('Int64')
orders_clean['INV'] = pd.to_numeric(orders_clean['INV'], errors='coerce')
orders_clean['QTY'] = pd.to_numeric(orders_clean['QTY'], errors='coerce')

print('Total ordenes unificadas:', len(orders_clean))
print('Fechas nulas despues de limpieza:', int(orders_clean['order_date'].isna().sum()))
print('Catalogos nulos despues de limpieza:', int(orders_clean['CATALOG'].isna().sum()))
print('PCODE nulos despues de limpieza:', int(orders_clean['PCODE'].isna().sum()))


Total ordenes unificadas: 7710
Fechas nulas despues de limpieza: 0
Catalogos nulos despues de limpieza: 3
PCODE nulos despues de limpieza: 11


In [21]:
quality_summary = pd.DataFrame({
    'metric': [
        'rows_total',
        'null_order_date',
        'null_catalog',
        'null_pcode',
        'invalid_qty',
        'invalid_inv'
    ],
    'value': [
        len(orders_clean),
        int(orders_clean['order_date'].isna().sum()),
        int(orders_clean['CATALOG'].isna().sum()),
        int(orders_clean['PCODE'].isna().sum()),
        int((orders_clean['QTY'].isna() | (orders_clean['QTY'] <= 0)).sum()),
        int(orders_clean['INV'].isna().sum())
    ]
})

display(quality_summary)

print('Frecuencia de errores CATALOG (valores no canonicos en origen):')
raw_catalog = pd.concat([
    catalog_df['CATALOG'].astype(str).str.strip(),
    web_df['CATALOG'].astype(str).str.strip()
], ignore_index=True)
error_catalog = ~raw_catalog.str.lower().isin([c.lower() for c in CANONICAL_CATALOGS])
print('Errores CATALOG:', int(error_catalog.sum()), 'de', len(raw_catalog), 'registros')


,metric,value
0,rows_total,7710
1,null_order_date,0
2,null_catalog,3
3,null_pcode,11
4,invalid_qty,10
5,invalid_inv,0


Frecuencia de errores CATALOG (valores no canonicos en origen):
Errores CATALOG: 341 de 7710 registros


In [22]:
# Re-analisis rapido post limpieza
print('Distribucion CATALOG limpio:')
display(orders_clean['CATALOG'].value_counts(dropna=False))

print('Rango de fechas limpio:')
print('Min:', orders_clean['order_date'].min())
print('Max:', orders_clean['order_date'].max())

print('QTY estadisticos:')
display(orders_clean['QTY'].describe())


Distribucion CATALOG limpio:


CATALOG
Sports          2294
Pets            1875
Toys            1629
Gardening       1151
Software         577
Collectibles     181
NaN                3
Name: count, dtype: int64

Rango de fechas limpio:
Min: 1997-01-01 00:00:00
Max: 2001-08-09 00:00:00
QTY estadisticos:


count    7700.000000
mean        1.779870
std        15.434563
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max      1353.000000
Name: QTY, dtype: float64

In [23]:
# Export opcional para fase ETL
orders_clean.to_csv('orders_clean.csv', index=False)
products_df.to_csv('products_clean.csv', index=False)
print('Archivos generados: orders_clean.csv, products_clean.csv')


Archivos generados: orders_clean.csv, products_clean.csv
